In [2]:
import pandas as pd

sales_data = pd.DataFrame({
    "id": [1,2,3,4,5,6],
    "region": ["North","South","North","East","West","South"],
    "product": ["Laptop","Phone","Tablet","Laptop","Phone","Tablet"],
    "month": ["Jan","Jan","Feb","Feb","Mar","Mar"],
    "sales": [800,500,300,850,520,310],
    "tags": [["electronics","office"], ["electronics"], ["gadgets"], ["electronics","premium"], ["gadgets","discount"], ["gadgets"]]
})
df=pd.DataFrame(sales_data)
df=df.set_index("id")

# Pivot Table

In [4]:
# Create a pivot table showing average sales by region and product.
pd.pivot_table(df,values="sales",index="region",columns="product",aggfunc="mean")


product,Laptop,Phone,Tablet
region,,,
East,850.0,NaN,NaN
North,800.0,NaN,300.0
South,NaN,500.0,310.0
West,NaN,520.0,NaN


In [ ]:
# Add margins(totals) to see overall averages.
pd.pivot_table(df,values="sales",index="region",columns="product",aggfunc="mean",margins=True)

product,Laptop,Phone,Tablet,All
region,,,,
East,850.0,NaN,NaN,850.000000
North,800.0,NaN,300.0,550.000000
South,NaN,500.0,310.0,405.000000
West,NaN,520.0,NaN,520.000000
All,825.0,510.0,305.0,546.666667


# Crosstab

In [10]:
# Build a crosstab of region vs product showing frequency counts.
pd.crosstab(df["region"],df["product"])

product,Laptop,Phone,Tablet
region,,,
East,1,0,0
North,1,0,1
South,0,1,1
West,0,1,0


In [ ]:
# Normalize by row to get proportions.
pd.crosstab(df["region"],df["product"],normalize="index")


product,Laptop,Phone,Tablet
region,,,
East,0.5,0.0,0.0
North,0.5,0.0,0.5
South,0.0,0.5,0.5
West,0.0,0.5,0.0


# MultiIndex

In [ ]:
# Set a MultiIndex with region and month.
multi_df=df.set_index(["region","month"])
print(multi_df)

             product  sales                    tags
region month                                       
North  Jan    Laptop    800   [electronics, office]
South  Jan     Phone    500           [electronics]
North  Feb    Tablet    300               [gadgets]
East   Feb    Laptop    850  [electronics, premium]
West   Mar     Phone    520     [gadgets, discount]
South  Mar    Tablet    310               [gadgets]


In [19]:
# Select all sales for "North" in "Feb".
print(multi_df.loc[("North","Feb")])

product       Tablet
sales            300
tags       [gadgets]
Name: (North, Feb), dtype: object


# Stack/Unstack

In [26]:
result=pd.pivot_table(df,values="sales",index="region",columns="product",aggfunc="mean")
print(result)

product  Laptop  Phone  Tablet
region                        
East      850.0    NaN     NaN
North     800.0    NaN   300.0
South       NaN  500.0   310.0
West        NaN  520.0     NaN


In [31]:
# Stack the pivot table from Q1 so products become a row level.
# stacked=result.stack().dropna()
stacked=result.stack()
print(stacked)

region  product
East    Laptop     850.0
        Phone        NaN
        Tablet       NaN
North   Laptop     800.0
        Phone        NaN
        Tablet     300.0
South   Laptop       NaN
        Phone      500.0
        Tablet     310.0
West    Laptop       NaN
        Phone      520.0
        Tablet       NaN
dtype: float64


In [28]:
# Then unstack it back to wide format.
unstacked=stacked.unstack()
print(unstacked)

product  Laptop  Phone  Tablet
region                        
East      850.0    NaN     NaN
North     800.0    NaN   300.0
South       NaN  500.0   310.0
West        NaN  520.0     NaN


# Melt

In [32]:
# Melt the DF so that product and sales become long format under one column.
melted=result.reset_index().melt(
    id_vars="region",
    value_vars=["Laptop","Phone","Tablet"],
    var_name="Product",
    value_name="Sales"

)
print(melted)

   region Product  Sales
0    East  Laptop  850.0
1   North  Laptop  800.0
2   South  Laptop    NaN
3    West  Laptop    NaN
4    East   Phone    NaN
5   North   Phone    NaN
6   South   Phone  500.0
7    West   Phone  520.0
8    East  Tablet    NaN
9   North  Tablet  300.0
10  South  Tablet  310.0
11   West  Tablet    NaN


In [33]:
# Rename the varibale column to "Category"
melted=melted.rename(columns={"Product":"Category"})
print(melted)

   region Category  Sales
0    East   Laptop  850.0
1   North   Laptop  800.0
2   South   Laptop    NaN
3    West   Laptop    NaN
4    East    Phone    NaN
5   North    Phone    NaN
6   South    Phone  500.0
7    West    Phone  520.0
8    East   Tablet    NaN
9   North   Tablet  300.0
10  South   Tablet  310.0
11   West   Tablet    NaN


# Pivot(simple reshape)

In [ ]:
# Pivot the DF so that month becomes columns, region becomes rows, and values are sales.
pd.pivot_table(df,index="region",columns="month",values="sales")

month,Feb,Jan,Mar
region,,,
East,850.0,NaN,NaN
North,300.0,800.0,NaN
South,NaN,500.0,310.0
West,NaN,NaN,520.0


# Explode

In [35]:
# Explode the tags column so each tag gets its own row.
exploded=df.explode("tags")
print(exploded)

   region product month  sales         tags
id                                         
1   North  Laptop   Jan    800  electronics
1   North  Laptop   Jan    800       office
2   South   Phone   Jan    500  electronics
3   North  Tablet   Feb    300      gadgets
4    East  Laptop   Feb    850  electronics
4    East  Laptop   Feb    850      premium
5    West   Phone   Mar    520      gadgets
5    West   Phone   Mar    520     discount
6   South  Tablet   Mar    310      gadgets


In [38]:
# Count how many times each tag appears.
tag_counts=exploded["tags"].value_counts()
print(tag_counts)

tags
electronics    3
gadgets        3
office         1
premium        1
discount       1
Name: count, dtype: int64
